In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets scikit-learn ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


# 1. Загрузка данных

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(uciml_id=222):
    dataset = fetch_ucirepo(id=uciml_id)
    X = dataset.data.features
    y = dataset.data.targets
    df = pd.concat([X, y], axis=1)

    feature_names = X.columns.to_list()
    target_name = 'y'

    # Преобразование целевой переменной в бинарный формат
    le = LabelEncoder()
    df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, seed=42):

    # Разделение на train/test в соотношении 8 к 2
    train_df, test_df = train_test_split(
          df,
          test_size=test_size,
          random_state=seed,
          stratify=df[target_name]
      )

    return train_df, test_df

df, feature_names, target_name = load_dataset()
train_df, test_df = split_dataset(df, target_name)

df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,0
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,0
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,0
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  int64 
dtypes: int64(8), object(9)
memory usage: 5.9+ MB


In [ ]:
for name, df in [("train", train_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  no  (0): {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  yes (1): {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 36168):
  no  (0): 31937 — 88.3%
  yes (1):  4231 — 11.7%

test (всего: 9043):
  no  (0):  7985 — 88.3%
  yes (1):  1058 — 11.7%


# 2. Вспомогательные функции

In [ ]:
# Преобразование строки DataFrame в текстовое описание

def row_to_text_template(row, feature_names, target_name, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]
        # Форматирования значений в зависимости от типа
        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts )

    if include_target:
        target_value = "yes" if row[target_name] == 1 else "no"
        text += f" -> subscription: {target_value}"

    return text

print(row_to_text_template(train_df.iloc[0], feature_names, target_name, 1))
train_df.head(1)

The value of age is 36. The category of job is technician. The category of marital is divorced. The category of education is secondary. The category of default is no. The value of balance is 861. The category of housing is no. The category of loan is no. The category of contact is telephone. The value of day_of_week is 29. The category of month is aug. The value of duration is 140. The value of campaign is 2. The value of pdays is -1. The value of previous is 0. The value of poutcome is nan. -> subscription: no


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
24001,36,technician,divorced,secondary,no,861,no,no,telephone,29,aug,140,2,-1,0,NaN,0


In [ ]:
def parse_prediction(response):
    response = response.lower().strip()

    # Удаление пунктуаций
    response = response.rstrip('.,!?')

    # Поиск "yes" или "no" в ответе
    if response == "yes" or response.startswith("yes"):
        return 1
    elif response == "no" or response.startswith("no"):
        return 0
    elif "yes" in response.split():
        return 1
    elif "no" in response.split():
        return 0
    else:
        # Если не можем распознать, возвращаем случайное
        print(f"Warning: Could not parse response: '{response}'")
        return random.randint(0, 1)



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec


# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}...")


Пример текста:
The value of age is 40. The category of job is blue-collar. The category of marital is married. The category of education is primary. The category of default is no. The value of balance is 640. The category of housing is yes. The category of loan is yes. The value of contact is nan. The value of day...


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}...")


Пример текста:
The value of age is 40. The category of job is blue-collar. The category of marital is married. The category of education is primary. The category of default is no. The value of balance is 640. The category of housing is yes. The category of loan is yes. The value of contact is nan. The value of day...


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 2.5 7B Instruct
model_name = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device # Explicitly force all model layers to the detected device (GPU if available)
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Использовано памяти: 30.63 GB


In [ ]:
def create_prompt(row, feature_names, target_name, few_shot_examples=None):

    system_prompt = (
        "You are a classifier. Predict whether a bank client will subscribe: "
        "'yes' if they will subscribe, or 'no' if they won't. "
        "Answer with only one word: 'yes' or 'no'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"Client information: "
            f"{row_to_text_template(row, feature_names, target_name)}\n"
            f"Will this client subscribe to a term deposit?"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name)
            ex_target = "yes" if ex[target_name] == 1 else "no"
            messages.append({"role": "user",      "content": f"Client information: {ex_text}\nWill this client subscribe?"})
            messages.append({"role": "assistant", "content": ex_target})

        client_text = row_to_text_template(row, feature_names, target_name)
        messages.append({"role": "user", "content": f"Client information: {client_text}\nWill this client subscribe?"})

    return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True # добавление role assistant
            )

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

def predict_single(prompt, max_new_tokens=10):
    # Получение предсказания для одного примера

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Декодирирование только новых токенов
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip().lower()


# Тестирование инференса
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name)
test_response = predict_single(test_prompt)
test_prediction = parse_prediction(test_response)

print(f"Ответ модели: '{test_response}'")
print(f"Распознанное предсказание: {test_prediction}")
print(f"Истинная метка: {test_df.iloc[0][target_name]}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Ответ модели: 'no'
Распознанное предсказание: 0
Истинная метка: 0


In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, model, tokenizer, device, max_new_tokens=10):
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Текст ответа
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Вероятность из logits
    first_token_logits = outputs.scores[0][0]

    yes_ids = tokenizer.encode("yes", add_special_tokens=False)
    no_ids  = tokenizer.encode("no",  add_special_tokens=False)
    # print(f"[DEBUG] yes token ids: {yes_ids}, no token ids: {no_ids}")

    yes_logit = first_token_logits[yes_ids[0]]
    no_logit = first_token_logits[no_ids[0]]

    probs = F.softmax(torch.stack([yes_logit, no_logit]), dim=0)
    prob_yes = probs[0].item()

    return response, prob_yes

test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name)
test_response, test_prob = predict_single_with_prob(test_prompt, model, tokenizer, device)
test_pred = parse_prediction(test_response)

print(f"Ответ модели: '{test_response}'")
print(f"Распознанное предсказание: {test_prediction}")
print(f"Истинная метка: {test_df.iloc[0][target_name]}")

Ответ модели: 'no'
Распознанное предсказание: 0
Истинная метка: 0


# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
# @title
df_majority = train_df[train_df[target_name] == 0]
df_minority = train_df[train_df[target_name] == 1]

seed = 42
print(f"До балансировки:")
print(f"no:  {len(df_majority)}")
print(f"yes: {len(df_minority)}")

# Oversample to match
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=seed
)

train_df_balanced = pd.concat([df_majority, df_minority_upsampled])
train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nПосле балансировки:")
print(train_df_balanced[target_name].value_counts())

До балансировки:
no:  31937
yes: 4231

После балансировки:
y
1    31937
0    31937
Name: count, dtype: int64


In [ ]:
# Создание текстов для обучения из train_df_balanced
system_prompt = (
        "You are a classifier. Predict whether a bank client will subscribe: "
        "'yes' if they will subscribe, or 'no' if they won't. "
        "Answer with only one word: 'yes' or 'no'."
    )
train_texts = []

print(f"Подготовка {len(train_df_balanced)} примеров...")
for idx, (_, row) in enumerate(tqdm(train_df_balanced.iterrows(), total=len(train_df_balanced))):
    input_text = row_to_text_template(row, feature_names, target_name)
    target = "yes" if row[target_name] == 1 else "no"

    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user", "content": f"Client information: {input_text}\nWill this client subscribe to a term deposit?"},
        {"role": "assistant", "content": target}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    train_texts.append(text)

# Создание Dataset
train_dataset = Dataset.from_dict({"text": train_texts})


Подготовка 63874 примеров...


  0%|          | 0/63874 [00:00<?, ?it/s]

In [ ]:
# Токенизация
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(f"\n Подготовлено {len(tokenized_dataset)} примеров для обучения")

Map:   0%|          | 0/63874 [00:00<?, ? examples/s]


 Подготовлено 63874 примеров для обучения


In [ ]:
# Параметры обучения
num_epochs = 1
batch_size = 16
output_dir = "/content/drive/MyDrive/finetuned_qwen_lora"

model_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    dtype=torch.bfloat16,
)

# Настройка LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

training_args_lora = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    optim="adamw_torch_fused",
    warmup_steps=50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    report_to="none",
    dataloader_num_workers=4,
    group_by_length=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/finetuned_qwen_lora"

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Обучение
print(f"\nНачинаем обучение на {num_epochs} эпох")

train_start_time = time.time()
trainer_lora.train()
training_time = time.time() - train_start_time

print(f"training_time: {training_time}")

trainer_lora.save_model(output_dir)
print(f"Модель сохранена в {output_dir}")


Mounted at /content/drive

Начинаем обучение на 1 эпох


Step,Training Loss
50,0.734369
100,0.178073
150,0.175704
200,0.173297
250,0.170782
300,0.171170
350,0.171900
400,0.171017
450,0.171180
500,0.170376


training_time: 2978.014457464218
Модель сохранена в /content/drive/MyDrive/finetuned_qwen_lora


In [ ]:
# from google.colab import drive

# drive.mount('/content/drive')

# drive_output_dir = "/content/drive/MyDrive/finetuned_qwen_lora"

# # Сохранение LoRA weights + tokenizer
# trainer_lora.save_pretrained(drive_output_dir)
# tokenizer.save_pretrained(drive_output_dir)

# print(f"Сохранено: {drive_output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


AttributeError: 'Trainer' object has no attribute 'save_pretrained'

In [ ]:
from google.colab import drive
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

drive.mount('/content/drive')
model_name     = "Qwen/Qwen2.5-7B-Instruct"
drive_output_dir = "/content/drive/MyDrive/finetuned_qwen_lora"
device = "cuda" if torch.cuda.is_available() else "cpu"

model_finetuned = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
)
model_finetuned = PeftModel.from_pretrained(model_finetuned, drive_output_dir)
model_finetuned.eval()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [ ]:
# Проверка качества обучения Fine-tuned модели

test_samples = {
    'yes': test_df[test_df[target_name] == 1].sample(n=10, random_state=42),
    'no': test_df[test_df[target_name] == 0].sample(n=10, random_state=42)
}

# Проверка распределения вероятностей
print("\nРаспределение вероятностей")
all_probs = []
for _, row in test_df.sample(n=100, random_state=42).iterrows():
    prompt = create_prompt(row, feature_names, target_name)
    _, prob = predict_single_with_prob(prompt, model_finetuned, tokenizer, device)
    all_probs.append(prob)

all_probs = np.array(all_probs)
print(f"P(yes) - Mean: {all_probs.mean():.4f}, Std: {all_probs.std():.4f}")
print(f"P(yes) - Min: {all_probs.min():.4f}, Max: {all_probs.max():.4f}")
print(f"P(yes) - Median: {np.median(all_probs):.4f}")

# Признаки проблемы:
# - Mean ~0.05: все предсказания смещены в сторону "no"
# - Std ~0.01: почти нет вариации → модель не обучилась
# - Max < 0.2: модель ни разу не предсказала "yes" с уверенностью

# Нормальные значения:
# - Mean ~0.12 (соответствует распределению данных)
# - Std ~0.15-0.25 (разнообразные вероятности)
# - Max > 0.7 (некоторые примеры предсказаны с высокой уверенностью)


Распределение вероятностей
P(yes) - Mean: 0.2495, Std: 0.3703
P(yes) - Min: 0.0001, Max: 0.9807
P(yes) - Median: 0.0122


In [ ]:
# Предсказания на валидационной выборке
y_true_fine = []
y_pred_fine = []
y_prob_fine = []

start_time = time.time()

print(f"\nОбработка {len(test_df)} примеров...")
for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name)
    response, prob = predict_single_with_prob(prompt, model_finetuned, tokenizer, device)
    prediction = parse_prediction(response)

    y_true_fine.append(row[target_name])
    y_pred_fine.append(prediction)
    y_prob_fine.append(prob)

fine_tuned_time = time.time() - start_time

print(f"fine_tuned_time: {fine_tuned_time}")
print(f"fine_tuned_time / len(y_true_fine) {fine_tuned_time / len(y_true_fine)}")



Обработка 9043 примеров...


  0%|          | 0/9043 [00:00<?, ?it/s]

fine_tuned_time: 1410.8730442523956
fine_tuned_time / len(y_true_fine) 0.15601825105080125


In [ ]:
# Вычисляем метрики

roc_fine, f1_fine, acc_fine, pr_fine, rec_fine = compute_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine)
)
print("Результаты FINE-SHOT:")
print(f"ROC AUC: {roc_fine}")
print(f"F1 Score: {f1_fine}")
print(f"Accuracy: {acc_fine}")
print(f"Precision: {pr_fine}")
print(f"Recall: {rec_fine}")


Результаты FINE-SHOT:
ROC AUC: 0.9288262609595259
F1 Score: 0.5739938080495356
Accuracy: 0.847838106822957
Precision: 0.42679558011049723
Recall: 0.8761814744801513


In [ ]:
fine_tuned_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_fine),
    np.array(y_pred_fine),
    np.array(y_prob_fine),
    n_iter=1000
)

print("\nРезультаты fine-tune (bootstrap метрики с доверительными интервалами):")
for key, value in fine_tuned_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты fine-tune (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.9289±0.0035
  F1: 0.5744±0.0106
  Accuracy: 0.8479±0.0038
  Precision: 0.4272±0.0108
  Recall: 0.8765±0.0100


ROC-AUC (0.928): научилась различать классы.

Accuracy (0.847): выше, чем у few-shot, и близко к zero-shot

Precision (0.427), Recall (0.876): Модель находит 81% потенциальных клиентов. Из тех, кому она говорит "yes", 46% действительно подпишутся. Это из-за обучения на сбалансированных данных (50:50), тогда как реальное распределение — 88:12.

ROC-AUC - state-of-the-art уровень. Находит 87% потенциальных клиентов (high recall). Конверсия 42% - почти в 4 раза выше базовой

Время обучения: ~ 50 мин.

Время тестирования: ~ 25 мин.

Использовано оперативная памяти графического процесса: 63/80GB

Графический процессор A100 80GB